# 00 — Feature Extraction
Run this whenever your dataset changes or a new fruit is added.
All downstream notebooks (`01_train`, `02_evaluate`, `03_predict`) depend on the artifacts saved here.

**Outputs saved to `pipeline_v2/artifacts/`:**
- `X_fused.npy` — shape (N, 1310)
- `y.npy` — freshness labels (0=fresh, 1=rotten)
- `fruit_type.npy` — fruit species per sample
- `dataset_hash.txt` — stable md5 fingerprint (skip re-extraction if unchanged)

In [1]:
import os, warnings, hashlib
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import cv2
from pathlib import Path
from tqdm import tqdm

import tensorflow as tf
from skimage.feature import graycomatrix, graycoprops
from scipy.stats import entropy

os.makedirs("pipeline_v2/artifacts", exist_ok=True)
os.makedirs("pipeline_v2/models",    exist_ok=True)

DATASET_ROOT = "DataSet"          # ← change if your folder is named differently
HASH_FILE    = "pipeline_v2/artifacts/dataset_hash.txt"
EXTS         = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("TF version:", tf.__version__)


TF version: 2.13.0


## Dataset fingerprint
Uses `hashlib.md5` — stable across Python sessions, unlike `hash()`.

In [2]:
def compute_dataset_hash(folder: str) -> str:
    """Stable md5 fingerprint of all image paths + file sizes."""
    file_info = []
    for root, _, files in os.walk(folder):
        for f in sorted(files):
            if Path(f).suffix.lower() in EXTS:
                path = os.path.join(root, f)
                file_info.append(f"{path}:{os.path.getsize(path)}")
    raw = "|".join(sorted(file_info)).encode()
    return hashlib.md5(raw).hexdigest()


current_hash = compute_dataset_hash(DATASET_ROOT)
print("Current hash:", current_hash)

saved_hash = ""
if os.path.exists(HASH_FILE):
    saved_hash = open(HASH_FILE).read().strip()
    print("Saved  hash:", saved_hash)

NEED_EXTRACT = (current_hash != saved_hash)
print("→ Re-extraction needed:", NEED_EXTRACT)


Current hash: edfc4dd5051e2c40bccfecfc19faf353
→ Re-extraction needed: True


## Handcrafted feature extractor  
Extracts **exactly 30 features**: RGB(6) + HSV(6) + LAB(6) + Texture(5) + Shape(6) + Dark(1).

In [3]:
def extract_handcrafted(img_bgr) -> np.ndarray:
    """Return a 30-d float32 feature vector from a BGR image."""
    img = cv2.resize(img_bgr, (224, 224)).astype(np.float32)

    # RGB (6)
    b, g, r = cv2.split(img)
    rgb = [r.mean(), g.mean(), b.mean(), r.std(), g.std(), b.std()]

    # HSV (6)
    hsv     = cv2.cvtColor(img.astype(np.uint8), cv2.COLOR_BGR2HSV)
    h, s, v = (hsv[:, :, i].astype(np.float32) for i in range(3))
    h_rad   = h * (2 * np.pi / 180)
    cx, sx  = np.cos(h_rad).mean(), np.sin(h_rad).mean()
    hsv_f   = [float(np.arctan2(sx, cx)), float(s.mean()), float(v.mean()),
               float(np.sqrt(-2 * np.log(np.sqrt(cx**2 + sx**2) + 1e-6))),
               float(s.std()), float(v.std())]

    # LAB (6)
    lab     = cv2.cvtColor(img.astype(np.uint8), cv2.COLOR_BGR2LAB)
    L, a, b2 = (lab[:, :, i].astype(np.float32) for i in range(3))
    lab_f   = [L.mean(), L.std(), a.mean(), a.std(), b2.mean(), b2.std()]

    # Texture (5)
    gray    = cv2.cvtColor(img.astype(np.uint8), cv2.COLOR_BGR2GRAY)
    lap     = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    glcm    = graycomatrix((gray // 8).astype(np.uint8), [1], [0],
                           levels=32, symmetric=True, normed=True)
    hist, _ = np.histogram(gray, bins=256, range=(0, 255), density=True)
    tex_f   = [lap,
               float(graycoprops(glcm, 'contrast')[0, 0]),
               float(graycoprops(glcm, 'energy')[0, 0]),
               float(graycoprops(glcm, 'homogeneity')[0, 0]),
               float(entropy(hist + 1e-6))]

    # Shape (6)
    _, th     = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    cnts, _   = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if cnts:
        c     = max(cnts, key=cv2.contourArea)
        area  = float(cv2.contourArea(c))
        peri  = float(cv2.arcLength(c, True))
        hull_a = float(cv2.contourArea(cv2.convexHull(c)))
        x_, y_, w_, h_ = cv2.boundingRect(c)
        shp_f = [area, peri,
                 4 * np.pi * area / (peri * peri + 1e-6),
                 area / (hull_a + 1e-6),
                 w_ / (h_ + 1e-6),
                 area / (w_ * h_ + 1e-6)]
    else:
        shp_f = [0.0] * 6

    # Dark pixel ratio (1)
    feats = rgb + hsv_f + lab_f + tex_f + shp_f + [float((gray < 50).mean())]
    assert len(feats) == 30, f"Handcrafted count error: expected 30, got {len(feats)}"
    return np.array(feats, dtype=np.float32)


## EfficientNetB0 backbone  
Loaded once, used for all images. Output: 1280-d pooled embedding.

In [4]:
backbone   = tf.keras.applications.EfficientNetB0(
    include_top=False, weights="imagenet", pooling="avg"
)
preprocess = tf.keras.applications.efficientnet.preprocess_input
print("EfficientNetB0 loaded — output dim: 1280")


EfficientNetB0 loaded — output dim: 1280


## Extract all images → X_fused (N, 1310)
Skipped automatically if the dataset hash has not changed.

In [5]:
if not NEED_EXTRACT:
    print("Dataset unchanged — loading cached features.")
    X          = np.load("pipeline_v2/artifacts/X_fused.npy")
    y          = np.load("pipeline_v2/artifacts/y.npy")
    fruit_type = np.load("pipeline_v2/artifacts/fruit_type.npy", allow_pickle=True)
    FRUITS     = sorted(np.unique(fruit_type).tolist())
    print(f"Loaded  X_fused: {X.shape}  |  y: {y.shape}  |  fruits: {FRUITS}")

else:
    import concurrent.futures, math

    BATCH_SIZE  = 64                               # lower to 32 if MemoryError
    MAX_WORKERS = min(8, os.cpu_count() or 4)

    # ── Step 1: collect all (path, label, fruit_name) ──────────────────────
    records = []
    for freshness_dir in sorted(Path(DATASET_ROOT).iterdir()):
        if not freshness_dir.is_dir():
            continue
        label = 0 if freshness_dir.name.lower().startswith("fresh") else 1
        for fruit_dir in sorted(freshness_dir.iterdir()):
            if not fruit_dir.is_dir():
                continue
            fruit_name = fruit_dir.name.lower().replace(" ", "_")
            for p in fruit_dir.rglob("*"):
                if p.suffix.lower() in EXTS:
                    records.append((p, label, fruit_name))

    N = len(records)
    if N == 0:
        raise RuntimeError(f"No images found under '{DATASET_ROOT}'. "
                           "Check DATASET_ROOT and EXTS.")
    print(f"Found {N} images across "
          f"{len(set(r[2] for r in records))} fruit classes. Extracting...")

    # ── Step 2: parallel image read + resize ───────────────────────────────
    def _load(rec):
        img = cv2.imread(str(rec[0]))
        return cv2.resize(img, (224, 224)) if img is not None else None

    print(f"[1/3] Reading images ({MAX_WORKERS} threads)...", flush=True)
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        loaded = list(tqdm(ex.map(_load, records), total=N,
                           desc="read", ncols=80))

    valid  = [(img, rec[1], rec[2])
              for img, rec in zip(loaded, records) if img is not None]
    n_skip = N - len(valid)
    if n_skip:
        print(f"  ⚠  Skipped {n_skip} unreadable images.")
    if len(valid) == 0:
        raise RuntimeError("All images failed to load. Check file paths and formats.")
    N = len(valid)
    imgs_raw, labels_raw, fruits_raw = zip(*valid)

    # ── Step 3: handcrafted features in parallel ───────────────────────────
    print(f"[2/3] Handcrafted features ({MAX_WORKERS} threads)...", flush=True)
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        hand_feats = list(tqdm(ex.map(extract_handcrafted, imgs_raw),
                               total=N, desc="handcrafted", ncols=80))
    hand_arr = np.array(hand_feats, dtype=np.float32)          # (N, 30)
    assert hand_arr.shape[1] == 30, f"Handcrafted dim: {hand_arr.shape[1]}"

    # ── Step 4: EfficientNet batch inference ───────────────────────────────
    print(f"[3/3] EfficientNet batch inference (batch={BATCH_SIZE})...",
          flush=True)
    imgs_np    = preprocess(
        np.stack(imgs_raw, axis=0).astype(np.float32)
    )                                                          # (N, 224, 224, 3)
    emb_blocks = []
    for i in tqdm(range(math.ceil(N / BATCH_SIZE)),
                  desc="batches", ncols=80):
        batch = imgs_np[i * BATCH_SIZE : (i + 1) * BATCH_SIZE]
        emb_blocks.append(backbone.predict(batch, verbose=0))
    emb_arr = np.concatenate(emb_blocks, axis=0).astype(np.float32)  # (N, 1280)
    assert emb_arr.shape[1] == 1280, f"Embedding dim: {emb_arr.shape[1]}"

    # ── Step 5: fuse and save ──────────────────────────────────────────────
    X          = np.concatenate([emb_arr, hand_arr], axis=1)   # (N, 1310)
    y          = np.array(labels_raw, dtype=np.int32)
    fruit_type = np.array(fruits_raw,  dtype=object)
    FRUITS     = sorted(np.unique(fruit_type).tolist())

    assert X.shape[1] == 1310, f"Expected 1310 features, got {X.shape[1]}"

    np.save("pipeline_v2/artifacts/X_fused.npy",    X)
    np.save("pipeline_v2/artifacts/y.npy",           y)
    np.save("pipeline_v2/artifacts/fruit_type.npy",  fruit_type)
    open(HASH_FILE, "w").write(current_hash)

    print(f"\nDone.  X: {X.shape}  |  y: {y.shape}")
    print(f"Fruits : {FRUITS}")
    print(f"Fresh  : {(y==0).sum()}  |  Rotten: {(y==1).sum()}")
    print("Saved → pipeline_v2/artifacts/  |  dataset_hash.txt updated")

Found 13795 images across 12 fruit classes. Extracting...
[1/3] Reading images (8 threads)...


read: 100%|██████████████████████████████| 13795/13795 [00:44<00:00, 307.33it/s]

[2/3] Handcrafted features (8 threads)...



handcrafted: 100%|████████████████████████| 13795/13795 [06:50<00:00, 33.57it/s]

[3/3] EfficientNet batch inference (batch=64)...



batches: 100%|████████████████████████████████| 216/216 [13:27<00:00,  3.74s/it]



Done.  X: (13795, 1310)  |  y: (13795,)
Fruits : ['freshapple', 'freshbanana', 'freshcapsicum', 'freshcarrot', 'freshcucumber', 'freshpotato', 'rottenapple', 'rottenbanana', 'rottencapsicum', 'rottencarrot', 'rottencucumber', 'rottenpotato']
Fresh  : 6673  |  Rotten: 7122
Saved → pipeline_v2/artifacts/  |  dataset_hash.txt updated
